# 02 — Classical Clustering & Cell Type Annotation

Now that we have QC'd data, we'll run the standard scRNA-seq analysis pipeline: normalize, find the
genes that actually vary across cells, reduce dimensionality, cluster, and figure out what each
cluster *is* biologically. This is the "classical" (pre-foundation-model) workflow that's been the
field standard for ~a decade, and it's the baseline we'll compare a foundation model against in
notebook 03.

In [ ]:
import os

# On Colab, opening a notebook file directly starts a fresh kernel whose working
# directory is /content, regardless of where (or whether) you previously cloned
# the repo in another notebook/cell. Relative paths like "../data/..." only
# resolve correctly if the kernel's cwd is this repo's notebooks/ directory, so
# make sure of that here before anything else runs. This is a no-op locally.
if os.path.basename(os.getcwd()) != "notebooks":
    for candidate in ("biohack-2026/notebooks", "notebooks"):
        if os.path.isdir(candidate):
            os.chdir(candidate)
            break
print("Working directory:", os.getcwd())

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=80, facecolor="white")

adata = sc.read_h5ad("../data/pbmc3k_qc.h5ad")
adata

## 1. Normalize and transform

Raw counts aren't directly comparable across cells — a cell with more total RNA will have higher
counts for every gene, which has nothing to do with biology. We normalize each cell to the same
total count, then log-transform (expression data is heavily right-skewed; log-transforming makes
variance more comparable across the expression range, which most downstream methods assume).

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

## 2. Find highly variable genes

Most genes are either uninformative housekeeping genes or just noise at this depth of sequencing.
Restricting to highly variable genes (HVGs) focuses the analysis on genes that actually distinguish
cell populations, and makes PCA much more meaningful.

In [ ]:
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
print(f"{adata.var.highly_variable.sum()} highly variable genes out of {adata.n_vars}")
sc.pl.highly_variable_genes(adata)

In [ ]:
adata.raw = adata  # save the full log-normalized matrix before subsetting to HVGs
adata = adata[:, adata.var.highly_variable].copy()

# Scale each gene to zero mean, unit variance, clipping extreme outliers
sc.pp.scale(adata, max_value=10)

## 3. PCA, neighbors, UMAP

PCA compresses thousands of genes into a handful of components that capture most of the
cell-to-cell variation. We then build a nearest-neighbor graph in PCA space and use it both for
clustering (Leiden) and for a 2D visualization (UMAP).

In [ ]:
sc.tl.pca(adata, svd_solver="arpack")
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)

In [ ]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)
sc.tl.umap(adata)

## 4. Leiden clustering

Leiden clustering finds densely-connected communities in the neighbor graph — groups of cells that
look more like each other than like the rest of the dataset. It does **not** know what a "cell
type" is; it just finds structure. We still have to interpret what each cluster represents.

In [ ]:
sc.tl.leiden(adata, resolution=0.5, flavor="igraph", n_iterations=2)
sc.pl.umap(adata, color=["leiden"], title="Leiden Clustering")

**Stop and look at the UMAP.** PCA + Leiden separates cell populations, but the cluster labels
("0", "1", "2"...) are meaningless on their own. To know whether cluster 0 is T cells or B cells, we
need to look at *which genes* distinguish each cluster, and compare them against genes we already
know mark specific cell types — that's the next step.

## 5. Rank marker genes per cluster

For each cluster, we statistically test which genes are most differentially expressed compared to
all other clusters. The top of that ranked list is a cluster's marker genes.

`rank_genes_groups` defaults to using `adata.raw` when it's set, which is the full
log-normalized gene set we saved before subsetting to HVGs above — so marker gene ranking isn't
limited to only the genes that passed the HVG filter.

In [ ]:
sc.tl.rank_genes_groups(adata, "leiden", method="t-test")
sc.pl.rank_genes_groups(adata, n_genes=20, sharey=False)

In [ ]:
top_markers = pd.DataFrame(adata.uns["rank_genes_groups"]["names"]).head(10)
top_markers

## 6. Manual annotation using canonical markers

This is where domain knowledge comes in. PBMCs have well-characterized marker genes from decades of
immunology research, e.g.:

| Marker gene(s) | Cell type |
|---|---|
| `CD3D`, `CD3E`, `IL32` | T cells |
| `CD8A` | CD8+ T cells |
| `NKG7`, `GNLY`, `GZMA` | NK cells |
| `CD79A`, `MS4A1`, `CD74` | B cells |
| `LYZ`, `S100A9`, `FTL` | Monocytes |
| `FCER1G`, `HLA-DPA1` | Dendritic cells |
| `PPBP`, `PF4` | Megakaryocytes / platelets |

Look at the top markers table above and the ranked-genes plot, match them against this table, and
fill in the mapping below. **The cluster numbers and best-matching cell type will vary depending on
the exact data and resolution** — look at your own output rather than assuming the dictionary below
is already correct for your run.

In [ ]:
# EDIT THIS based on what you see in the marker gene plot above
cluster_annotations = {
    "0": "CD4 T cells",
    "1": "NK cells / Cytotoxic T cells",
    "2": "B cells",
    "3": "Monocytes",
    "4": "Dendritic cells",
    "5": "Megakaryocytes",
}

adata.obs["cell_type"] = adata.obs["leiden"].map(cluster_annotations).astype("category")
sc.pl.umap(adata, color=["cell_type"], title="Manually Annotated Cell Types")

## 7. Save labeled data

We save the cell type labels (and the underlying PCA features) — notebook 03 will treat these
manual annotations as ground truth to evaluate against, and will compare PCA features against
foundation model embeddings on a downstream classification task.

In [ ]:
labels = adata.obs[["leiden", "cell_type"]].copy()
labels["cell_id"] = labels.index
labels.to_csv("../data/pbmc3k_cell_type_labels.csv", index=False)

pca_df = pd.DataFrame(adata.obsm["X_pca"], index=adata.obs_names)
pca_df.to_csv("../data/pbmc3k_pca_features.csv")

print("Saved labels for", len(labels), "cells across", labels.cell_type.nunique(), "cell types")
labels.cell_type.value_counts()

## Recap

- Classical scRNA-seq analysis: normalize -> HVGs -> PCA -> neighbor graph -> Leiden clustering ->
  marker genes -> manual annotation against known biology.
- Every step up to clustering is unsupervised and "knows" nothing about cell type. The actual
  labeling step depends entirely on a human matching marker genes against prior literature.
- This means classical annotation doesn't scale well: every new dataset, tissue, or species needs
  someone who already knows the relevant marker genes.

**Next:** `03_geneformer_foundation_model.ipynb` — replace the "human matches markers against
known biology" step with a pretrained single-cell foundation model, and quantitatively compare how
well its learned representations support cell type classification versus the classical PCA
features above, especially when very few labeled cells are available.